# How good is the model at word problems before finetuning?

In [1]:
!pip install transformers accelerate datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 10.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 7.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 8.2 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of huggingface-hub to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of huggingface-hub to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.9/73.9 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 26.0 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## Answer Checker

In [1]:
# basic solution checkers
import re
import multiprocessing as mp


def final_answer_formatting(generated_text):
    """ check whether the answer ends with
        'The final answer is: __'
        and returns the imputed answer

        formatting codes: 0 - no good,
                          1 - contains answer,
                          2 - ends with 'The final answer is: __'
    """
    output = {'good_formatting': 0, 'perfect_formatting': 0, 'answer': ''}

    last_line = generated_text.split('\n')[-1]

    last_line_numbers = re.findall('[0-9]+', last_line)

    if len(last_line_numbers) > 0:
        output['answer'] = last_line_numbers[-1]

        if last_line.startswith('The final answer is:'):
            output['perfect_formatting'] = 1
            output['good_formatting'] = 1
        elif 'answer' in last_line.lower():
            output['good_formatting'] = 1

    return output


def COT_formatting(generated_text):
    pass


def results_worker(queue, output):
    """ worker for answer checker """
    while True:
        results = queue.get()
        if results is None:  # signal that the testing is done
            break

        samples, ground_truth = results
        output['num_inputs'] += 1

        correct_count = 0
        for sample in samples:
            answer_check = final_answer_formatting(sample['generated_text'])
            for key in ['good_formatting', 'perfect_formatting']:
                output[key] += answer_check[key]

            if answer_check['answer'] == ground_truth:
                correct_count += 1

        if correct_count > 0:
            output['passOf8'] += 1
        if correct_count > 4:
            output['majorityOf8'] += 1


class AnswerChecker(object):
    """ Multiprocessing answer checker
    """
    def __init__(self):
        self.queue = mp.Queue()
        self.results = mp.Manager().dict()
        self.results['good_formatting'] = 0
        self.results['perfect_formatting'] = 0
        self.results['majorityOf8'] = 0
        self.results['passOf8'] = 0
        self.results['num_inputs'] = 0

        self.worker = mp.Process(target=results_worker, args=(self.queue, self.results))
        self.worker.start()

    def add_result(self, results):
        self.queue.put(results)

    def shutdown(self):
        self.queue.put(None)

    def summarize(self):
        results = {key: self.results[key] / self.results['num_inputs']
                   for key in self.results.keys()}
        results.pop('num_inputs')
        results['num_samples'] = self.results['num_inputs']

        results['good_formatting'] = results['good_formatting'] / 8
        results['perfect_formatting'] = results['perfect_formatting'] / 8

        return results


In [2]:
answer_checker = AnswerChecker()

## Model and dataset setup

In [3]:
from transformers import pipeline
model = pipeline("text-generation", model="meta-llama/Llama-3.2-1B-Instruct",
                 device_map='auto', do_sample=True, temperature=1.,
                 )

model.tokenizer.pad_token_id = model.model.config.eos_token_id[2]
model.generation_config.pad_token = model.tokenizer.pad_token
model.generation_config.pad_token_id = model.tokenizer.pad_token_id

model.tokenizer.padding_side = 'left'

Device set to use cuda:0


In [4]:
from datasets import load_dataset
questions = load_dataset("openai/gsm8k", "main")

prompt = "Please solve the following math problem, reasoning step by step and stating the final answer at the end.\nQuestion: {question}\n"

def process_datapoint(datapoint):
    answer = datapoint.pop('answer').split('####')[-1].strip()
    question = prompt.format(**datapoint) 
    return {'question': question, 'gt': answer}

questions = questions['train'].map(process_datapoint)

In [5]:
import torch
import gc

batch_size=32
with torch.no_grad():
    for i in range(len(questions) // batch_size + 1):
        print(f'Epoch {i} of {len(questions) // batch_size + 1}')
        batch = questions[i*batch_size:(i+1)*batch_size]
    
        outs = model(batch['question'], batch_size=batch_size, num_return_sequences=8)
        
        for result in zip(outs, batch['gt']):
            answer_checker.add_result(result)

        gc.collect()
        torch.cuda.empty_cache()

Epoch 0 of 234
Epoch 1 of 234
Epoch 2 of 234
Epoch 3 of 234
Epoch 4 of 234
Epoch 5 of 234
Epoch 6 of 234
Epoch 7 of 234
Epoch 8 of 234
Epoch 9 of 234


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Epoch 10 of 234
Epoch 11 of 234
Epoch 12 of 234
Epoch 13 of 234
Epoch 14 of 234
Epoch 15 of 234
Epoch 16 of 234
Epoch 17 of 234
Epoch 18 of 234
Epoch 19 of 234
Epoch 20 of 234
Epoch 21 of 234
Epoch 22 of 234
Epoch 23 of 234
Epoch 24 of 234
Epoch 25 of 234
Epoch 26 of 234
Epoch 27 of 234
Epoch 28 of 234
Epoch 31 of 234
Epoch 32 of 234
Epoch 33 of 234
Epoch 34 of 234
Epoch 35 of 234
Epoch 36 of 234
Epoch 37 of 234
Epoch 38 of 234
Epoch 39 of 234
Epoch 40 of 234
Epoch 41 of 234
Epoch 42 of 234
Epoch 43 of 234
Epoch 44 of 234
Epoch 45 of 234
Epoch 46 of 234
Epoch 47 of 234
Epoch 48 of 234
Epoch 49 of 234
Epoch 50 of 234
Epoch 51 of 234
Epoch 52 of 234
Epoch 53 of 234
Epoch 54 of 234
Epoch 55 of 234
Epoch 56 of 234
Epoch 57 of 234
Epoch 58 of 234
Epoch 59 of 234
Epoch 60 of 234
Epoch 61 of 234
Epoch 62 of 234
Epoch 63 of 234
Epoch 64 of 234
Epoch 65 of 234
Epoch 66 of 234
Epoch 67 of 234
Epoch 68 of 234
Epoch 69 of 234
Epoch 70 of 234
Epoch 71 of 234
Epoch 72 of 234
Epoch 73 of 234
Epoch 74

In [6]:
answer_checker.shutdown()
answer_checker.worker.join()
results = answer_checker.summarize()

In [7]:
print(results)

{'good_formatting': 0.5330690485748695, 'perfect_formatting': 0.25107052054061285, 'majorityOf8': 0.2743208885320487, 'passOf8': 0.779874213836478, 'num_samples': 7473}


epoch 3: 60%
epoch 187: 55%

In [11]:
import pickle

In [14]:
with open('data.pkl', 'wb') as file:
    pickle.dump(results, file)

In [15]:
!runpodctl send data.pkl

Runpod config file not found, please run `runpodctl config` to create it
Sending 'data.pkl' (132 B)       
Code is: 8178-bonjour-impact-love-5
On the other computer run

runpodctl receive 8178-bonjour-impact-love-5

Sending (->209.6.183.189:56065)
data.pkl 100% |████████████████████| (132/132 B, 49.582 kB/s)
